# Pipeline di Modellazione: Approccio Giornaliero

🇮🇹 [🇬🇧](04_modeling_day.ipynb)

## Predizione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook implementa la **pipeline di modellazione per l'approccio giornaliero**. Addestra
e valuta quattro classificatori sui dati giornalieri preprocessati, ottimizza il modello migliore
e confronta i risultati con il benchmark dello studio originale.

### Modelli

| Modello | Strategia sbilanciamento | Scaling |
|---------|--------------------------|---------|
| **Dummy** (stratificato) | Nessuna — baseline casuale | No |
| **Regressione Logistica** | `class_weight='balanced'` | Sì (StandardScaler) |
| **Random Forest** | `class_weight='balanced'` | No |
| **XGBoost** | `scale_pos_weight` | No |

### Pipeline

1. Caricamento degli split preprocessati da `data/processed/`
2. Cross-validation di tutti i modelli con GroupKFold a 5 fold (ADR-006)
3. Ottimizzazione degli iperparametri di XGBoost (RandomizedSearchCV)
4. Valutazione finale sul test set
5. Curve ROC/PR, matrice di confusione, tabella comparativa
6. Esperimento SMOTE (secondario, ADR-003)

**Benchmark dello studio**: AUC-ROC **0.724** (approccio giornaliero, XGBoost con bagging).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ATHLETE_ID_COL, INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.modeling.evaluate import (
    compute_metrics,
    create_comparison_table,
    find_optimal_threshold,
    plot_confusion_matrix,
    plot_pr_curves,
    plot_roc_curves,
)
from src.modeling.models import create_model
from src.modeling.train import (
    cross_validate_model,
    train_final_model,
    tune_hyperparameters,
)
from src.preprocessing.common import get_feature_target_groups
from src.preprocessing.day_preprocessor import transform_scaled
from src.preprocessing.io import load_scaler, load_splits, save_model
from src.utils.logging_config import setup_logging
from src.utils.plotting import PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

---

## 1. Caricamento Dati

Carichiamo gli split preprocessati train/test e lo scaler addestrato da `data/processed/`.
Questi sono stati salvati nel notebook 03 dopo la sostituzione dei valori sentinella, la
suddivisione a livello di atleta e l'addestramento dello scaler solo sui dati di training.

In [ ]:
# Load preprocessed splits and scaler
day_train, day_test = load_splits(prefix="day")
day_scaler = load_scaler(name="day_scaler")
feature_cols = get_feature_columns(day_train)

# Extract features, target, groups for cross-validation
X_train, y_train, groups_train = get_feature_target_groups(day_train, feature_cols)
X_test = day_test[feature_cols]
y_test = day_test[INJURY_COL]

# Scaled versions for Logistic Regression
X_train_scaled = transform_scaled(day_train, day_scaler, feature_cols)[feature_cols]
X_test_scaled = transform_scaled(day_test, day_scaler, feature_cols)[feature_cols]

# Class imbalance ratio (for XGBoost scale_pos_weight)
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
imbalance_ratio = n_neg / n_pos

print("Day approach — preprocessed data:")
print(f"  Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"  Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"  Athletes: {groups_train.nunique()} train, {day_test[ATHLETE_ID_COL].nunique()} test")
print("\nClass balance (train):")
print(f"  Negative: {n_neg:,} ({n_neg / len(y_train) * 100:.2f}%)")
print(f"  Positive: {n_pos:,} ({n_pos / len(y_train) * 100:.2f}%)")
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1")

---

## 2. Baseline: Classificatore Dummy

Un classificatore Dummy stratificato predice le etichette di classe rispettando la
distribuzione di training — serve come **verifica di base**: qualsiasi modello reale
deve superarlo. AUC-ROC atteso ≈ 0.50 (prestazione casuale).

In [ ]:
# --- Store all CV results for comparison ---
cv_results: dict[str, dict] = {}

# Dummy baseline
dummy_model = create_model("dummy")
dummy_cv = cross_validate_model(dummy_model, X_train, y_train, groups_train)

cv_results["Dummy"] = {
    metric: f"{dummy_cv[f'test_{metric}'].mean():.4f} ± {dummy_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results["Dummy"]["roc_auc_mean"] = dummy_cv["test_roc_auc"].mean()

print("Dummy Classifier — 5-fold GroupKFold CV:")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = dummy_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 3. Regressione Logistica

La Regressione Logistica è l'unico modello che **richiede lo scaling delle feature** — i
suoi coefficienti e la regolarizzazione L2 sono sensibili alla magnitudine. Utilizziamo lo
scaler addestrato sui dati di training nel notebook 03 (nessun data leakage).

In [ ]:
# Logistic Regression on SCALED features
logreg_model = create_model("logistic_regression")
logreg_cv = cross_validate_model(logreg_model, X_train_scaled, y_train, groups_train)

cv_results["LogReg"] = {
    metric: f"{logreg_cv[f'test_{metric}'].mean():.4f} ± {logreg_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results["LogReg"]["roc_auc_mean"] = logreg_cv["test_roc_auc"].mean()

print("Logistic Regression — 5-fold GroupKFold CV (scaled features):")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = logreg_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 4. Random Forest

I modelli basati su alberi suddividono in base all'ordine di rango delle feature, non alla
magnitudine — **non è necessario lo scaling**. Random Forest utilizza `class_weight='balanced'`
per aumentare il peso della classe minoritaria.

In [ ]:
# Random Forest on UNSCALED features
model_label = "Random Forest"
rf_model = create_model("random_forest")
rf_cv = cross_validate_model(rf_model, X_train, y_train, groups_train)

cv_results[model_label] = {
    metric: f"{rf_cv[f'test_{metric}'].mean():.4f} ± {rf_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results[model_label]["roc_auc_mean"] = rf_cv["test_roc_auc"].mean()

print("Random Forest — 5-fold GroupKFold CV:")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = rf_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 5. XGBoost

XGBoost gestisce lo sbilanciamento delle classi tramite `scale_pos_weight` = rapporto tra
campioni negativi e positivi. Questo equivale ad assegnare un peso di loss maggiore alla
classe minoritaria — l'analogo nel boosting di `class_weight='balanced'`.

In [ ]:
# XGBoost on UNSCALED features
xgb_model = create_model("xgboost", imbalance_ratio=imbalance_ratio)
xgb_cv = cross_validate_model(xgb_model, X_train, y_train, groups_train)

cv_results["XGBoost"] = {
    metric: f"{xgb_cv[f'test_{metric}'].mean():.4f} ± {xgb_cv[f'test_{metric}'].std():.4f}"
    for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]
}
cv_results["XGBoost"]["roc_auc_mean"] = xgb_cv["test_roc_auc"].mean()

print(f"XGBoost — 5-fold GroupKFold CV (scale_pos_weight={imbalance_ratio:.1f}):")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = xgb_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

---

## 6. Confronto Cross-Validation

Confronto affiancato di tutti i modelli sulla metrica primaria (AUC-ROC) e
sulle metriche secondarie. Il benchmark dello studio (0.724) è mostrato come riferimento.

In [ ]:
# Build comparison table
cv_display = {
    model: {k: v for k, v in metrics.items() if k != "roc_auc_mean"}
    for model, metrics in cv_results.items()
}
cv_table = pd.DataFrame(cv_display).T
cv_table.index.name = "Model"
print("Cross-Validation Results (mean ± std across 5 folds):\n")
print(cv_table.to_string())

In [ ]:
# Bar chart: AUC-ROC comparison
PAPER_BENCHMARK_DAY = 0.724

model_names = list(cv_results.keys())
auc_means = [cv_results[m]["roc_auc_mean"] for m in model_names]

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE["neutral"] if m == "Dummy" else PALETTE["primary"] for m in model_names]
bars = ax.bar(model_names, auc_means, color=colors, edgecolor="white", width=0.6)

# Paper benchmark line
ax.axhline(PAPER_BENCHMARK_DAY, color=PALETTE["negative"], linestyle="--", linewidth=1.5,
           label=f"Paper benchmark ({PAPER_BENCHMARK_DAY})")

for bar, val in zip(bars, auc_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", fontweight="bold", fontsize=10)

ax.set_ylabel("AUC-ROC (CV mean)")
ax.set_title("Cross-Validation AUC-ROC — Day Approach", fontweight="bold")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
sns.despine()

save_figure(fig, "04_cv_auc_comparison", subdir="modeling", close=False)
plt.show()
plt.close(fig)

### Interpretazione

- **Dummy ≈ 0.50**: conferma l'assenza di information leakage — il guess casuale è casuale.
- Tutti i modelli reali dovrebbero superare sostanzialmente la baseline.
- Il benchmark dello studio (0.724) è stato ottenuto con un **XGBoost con bagging** — un
  ensemble più complesso del nostro singolo XGBoost. Avvicinarsi a quel benchmark
  con un singolo modello è un buon risultato.

---

## 7. Ottimizzazione degli Iperparametri (XGBoost)

Ottimizziamo XGBoost in quanto modello primario dello studio e con lo spazio di
iperparametri più ricco. `RandomizedSearchCV` con 30 iterazioni esplora
lo spazio dei parametri in modo efficiente.

> **Nota per la produzione**: `n_iter=30` è usato qui per la fattibilità su M1 locale.
> In ambienti di produzione, aumentare a **50+ iterazioni** (o usare l'ottimizzazione
> bayesiana) per un'esplorazione più approfondita.

In [ ]:
# Hyperparameter search space
param_distributions: dict = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5],
}

# Base model with imbalance ratio
xgb_base = create_model("xgboost", imbalance_ratio=imbalance_ratio)

# Tune — n_iter=30 for local feasibility (production: 50+)
xgb_tuned = tune_hyperparameters(
    xgb_base, X_train, y_train, groups_train,
    param_distributions=param_distributions,
    n_iter=30,
)

print("\nBest hyperparameters:")
for param, value in sorted(xgb_tuned.get_params().items()):
    if param in param_distributions:
        print(f"  {param}: {value}")

---

## 8. Valutazione Finale sul Test Set

Riaddestriamo ogni modello sull'**intero set di training** e valutiamo sul test set
riservato. Questa è la misura definitiva delle prestazioni — la cross-validation ha guidato
la selezione del modello, ma il test set non è mai stato visto durante il training o il tuning.

Per il modello migliore (XGBoost ottimizzato), troviamo anche la **soglia di classificazione
ottimale** che massimizza l'F1 sulle predizioni del training set, e poi la applichiamo
al test set per evitare data leakage nella selezione della soglia.

In [ ]:
# Train all models on full training set and collect test predictions
test_predictions: dict[str, tuple[np.ndarray, np.ndarray]] = {}
test_metrics: dict[str, dict[str, float]] = {}

xgb_label = "XGBoost (tuned)"

models_to_evaluate = {
    "Dummy": (create_model("dummy"), X_train, X_test),
    "LogReg": (create_model("logistic_regression"), X_train_scaled, X_test_scaled),
    "Random Forest": (create_model("random_forest"), X_train, X_test),
    "XGBoost (default)": (create_model("xgboost", imbalance_ratio=imbalance_ratio), X_train, X_test),
    xgb_label: (xgb_tuned, X_train, X_test),
}

for name, (model, X_tr, X_te) in models_to_evaluate.items():
    # XGBoost tuned is already fitted by RandomizedSearchCV, but we refit on full train
    fitted = train_final_model(model, X_tr, y_train)
    y_prob = fitted.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    test_predictions[name] = (np.array(y_test), y_prob)
    test_metrics[name] = compute_metrics(np.array(y_test), y_pred, y_prob)

# Display test metrics
test_table = create_comparison_table(test_metrics)
print("Test Set Evaluation (threshold = 0.5):\n")
print(test_table[["auc_roc", "auc_pr", "recall", "precision", "f1", "brier_score"]].to_string())

In [ ]:
# Optimal threshold — selected on training set to avoid test leakage
y_true_test, y_prob_best = test_predictions[xgb_label]

# Compute train predictions for threshold selection (no test leakage)
y_prob_train_best = xgb_tuned.predict_proba(X_train)[:, 1]
optimal_threshold = find_optimal_threshold(np.array(y_train), y_prob_train_best, metric="f1")

y_pred_optimal = (y_prob_best >= optimal_threshold).astype(int)
metrics_optimal = compute_metrics(y_true_test, y_pred_optimal, y_prob_best)

print(f"\n{xgb_label} — Train-selected threshold: {optimal_threshold:.2f}")
print("\nMetrics at optimal threshold vs default (0.5):")
print(f"  {'Metric':>15s}  {'Optimal':>10s}  {'Default':>10s}")
print(f"  {'─' * 15}  {'─' * 10}  {'─' * 10}")
for metric in ["recall", "precision", "f1", "specificity"]:
    opt_val = metrics_optimal[metric]
    def_val = test_metrics[xgb_label][metric]
    print(f"  {metric:>15s}  {opt_val:>10.4f}  {def_val:>10.4f}")

---

## 9. Curve ROC e PR

Le curve ROC mostrano il trade-off tra tasso di veri positivi e tasso di falsi positivi
attraverso tutte le soglie. Le curve PR sono più informative per dataset sbilanciati
perché si concentrano sulla classe minoritaria (precisione = TP / (TP + FP)).

In [ ]:
# ROC curves
fig_roc = plot_roc_curves(test_predictions, save_path=Path("modeling/04_roc_curves"))
# Add paper benchmark annotation
fig_roc.axes[0].annotate(
    f"Paper benchmark: AUC = {PAPER_BENCHMARK_DAY}",
    xy=(0.55, 0.05), fontsize=10, fontstyle="italic",
    color=PALETTE["negative"],
)
plt.show()
plt.close(fig_roc)

# PR curves
fig_pr = plot_pr_curves(test_predictions, save_path=Path("modeling/04_pr_curves"))
plt.show()
plt.close(fig_pr)

---

## 10. Matrice di Confusione

La matrice di confusione per il modello migliore (XGBoost ottimizzato) alla soglia ottimale
mostra le decisioni di classificazione effettive. Nella predizione degli infortuni, i **falsi negativi**
(infortuni mancati) sono più costosi dei **falsi positivi** (cautela non necessaria).

In [ ]:
# Confusion matrix at optimal threshold
fig_cm = plot_confusion_matrix(
    y_true_test, y_pred_optimal,
    title=f"XGBoost (Tuned) — Day Approach (threshold={optimal_threshold:.2f})",
    save_path=Path("modeling/04_confusion_matrix"),
)
plt.show()
plt.close(fig_cm)

print("\nConfusion matrix breakdown:")
print(f"  True Negatives:  {metrics_optimal['tn']:,}")
print(f"  False Positives: {metrics_optimal['fp']:,}")
print(f"  False Negatives: {metrics_optimal['fn']:,}")
print(f"  True Positives:  {metrics_optimal['tp']:,}")

---

## 11. Esperimento SMOTE (Secondario)

Come esperimento secondario (ADR-003), testiamo **SMOTE** (Synthetic Minority
Oversampling Technique) come alternativa alla ponderazione delle classi. SMOTE genera
campioni positivi sintetici durante il training per bilanciare le classi.

Punti chiave:
- SMOTE è applicato **all'interno di ogni fold della CV** (solo sui dati di training — mai sulla validazione)
- Impostiamo `scale_pos_weight=1.0` poiché SMOTE bilancia già le classi
- Confrontiamo il risultato con l'approccio di ponderazione delle classi usato sopra

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# SMOTE + XGBoost pipeline (scale_pos_weight=1.0 since SMOTE balances classes)
smote_xgb = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_SEED)),
    ("model", create_model("xgboost", imbalance_ratio=1.0)),
])

smote_cv = cross_validate_model(smote_xgb, X_train, y_train, groups_train)

smote_auc = smote_cv["test_roc_auc"].mean()
weighting_auc = cv_results["XGBoost"]["roc_auc_mean"]

print("SMOTE + XGBoost — 5-fold GroupKFold CV:")
for metric in ["roc_auc", "average_precision", "recall", "precision", "f1"]:
    scores = smote_cv[f"test_{metric}"]
    print(f"  {metric:>20s}: {scores.mean():.4f} ± {scores.std():.4f}")

print("\nComparison:")
print(f"  Class weighting (XGBoost):  AUC-ROC = {weighting_auc:.4f}")
print(f"  SMOTE + XGBoost:            AUC-ROC = {smote_auc:.4f}")
print(f"  Difference:                 {smote_auc - weighting_auc:+.4f}")
print(f"\n→ ADR-003 {'confirmed' if weighting_auc >= smote_auc - 0.02 else 'needs review'}: "
      f"class weighting is {'competitive with' if abs(smote_auc - weighting_auc) < 0.02 else 'compared to'} SMOTE.")

---

## 12. Salvataggio del Modello Migliore

Salviamo il modello XGBoost ottimizzato per il riutilizzo nei notebook successivi
(Fase 5 — Interpretabilità SHAP, Fase 6 — Analisi di fairness).

In [ ]:
# Retrain tuned XGBoost on full training set and save
best_model = train_final_model(xgb_tuned, X_train, y_train)
save_model(best_model, name="day_best_model")

print("Best model saved: day_best_model.pkl")
print(f"  Type: {type(best_model).__name__}")
print(f"  Test AUC-ROC: {test_metrics[xgb_label]['auc_roc']:.4f}")
print(f"  Optimal threshold: {optimal_threshold:.2f}")

---

## 13. Riepilogo

### Flusso della pipeline

```
data/processed/day_train.parquet + day_test.parquet
  → load_splits("day")
  → Cross-validation: Dummy, LogReg (scalato), RF, XGBoost
  → Ottimizzazione XGBoost (RandomizedSearchCV, 30 iterazioni)
  → Valutazione finale sul test set
  → Salvataggio modello migliore → data/processed/day_best_model.pkl
```

### Risultati principali

1. **Tutti i modelli reali superano la baseline Dummy** — confermando che le feature
   contengono segnale predittivo per il rischio di infortunio
2. **XGBoost** è il modello con le migliori prestazioni, coerente con la scelta dello studio
3. **L'ottimizzazione degli iperparametri** ha migliorato (o mantenuto) l'AUC-ROC di XGBoost
   rispetto ai valori di default
4. **Ponderazione classi vs SMOTE**: ADR-003 validata — la ponderazione delle classi è più
   semplice e competitiva con SMOTE per questo dataset
5. **Confronto con il benchmark**: il nostro risultato con singolo XGBoost va valutato
   rispetto allo 0.724 dello studio (ottenuto con XGBoost con bagging — un ensemble più complesso)

### Invarianti verificate

- Nessun leakage tra atleti: GroupKFold CV e GroupShuffleSplit per lo split test
- Nessun data leakage: scaler addestrato solo sul training, SMOTE applicato solo all'interno dei fold della CV
- Riproducibilità: RANDOM_SEED=42 in tutto il processo

### Prossimi passi

→ **Notebook 05**: Modellazione — approccio settimanale (stessa pipeline sui dati settimanali)
→ **Notebook 06**: Analisi di interpretabilità SHAP sul modello salvato